# 17 — Regional specialisation and layering

This notebook presents the independently specified regional-specialisation screen. It does not refit models: the reusable script has already fitted the fold-local experts and persisted the evidence. The local test remains closed.

**Result:** the fixed 80% global / 20% regional Random Forest candidate reaches 81.469% mean accuracy, 0.156 percentage points below the accepted global vote, with zero fold wins. Hard routing loses 1.120 points. The global trees already use region and related geography; separate experts lose more through reduced pooling than they gain through specialisation.

## Course-aligned lifecycle

| Step | Application |
| --- | --- |
| 1. Define the goal and scope | Test regional specialisation beyond the accepted geography-aware ensemble. |
| 2. Gather the data | Reuse the validated labelled data; competition rows are not used for selection. |
| 3. Explore the data | Compare regional support, class shares, probabilities and OOF errors. |
| 4. Clean and preprocess the data | Fit accepted preprocessing separately inside every eligible outer-fold regional expert. |
| 5. Select and engineer features | Retain the accepted feature policy and route on the named region. |
| 6. Define the machine-learning task | Three-class nominal classification. |
| 7. Partition the data | Reuse the untouched local test and five frozen development folds. |
| 8. Select and train candidate methods | Test prior layers, hard regional RF routing and one fixed partial pool. |
| 9. Evaluate and interpret the results | Apply the established gate and inspect regional and probability diagnostics. |
| 10. Deploy and iterate | Not applicable: no candidate passes the gate. |

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd

STAGE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = STAGE_DIR / 'src'
PROJECT_DIR = STAGE_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

result = joblib.load(
    PROJECT_DIR / '.runtime' / 'regional-specialisation-screen'
    / 'regional-specialisation-screen.joblib'
)
result.candidate_summary

## Interpretation

The expert eligibility rule is fitted inside every outer fold: at least 500 regional training rows and 20 examples from every class. Twenty regions qualify in every fold. Dar es Salaam always falls back to the global model, including one fold whose regional training rows contain no repair example. Expert coverage is 98.69%.

The accepted probabilities already track the observed repair prevalence: Kigoma is 21.77% observed versus 21.82% predicted, and Dar es Salaam is 0.32% versus 0.38%. An unsmoothed regional-prior correction double-counts that signal and loses 1.206 accuracy points; smoothing equivalent to 10,000 national-prior observations still loses 0.120 points.

The full reasoning, pitfalls and next-step boundary are in [`../reports/regional-specialisation-and-layering.md`](../reports/regional-specialisation-and-layering.md).

In [ ]:
key_regions = ['Kigoma', 'Dar es Salaam']
result.regional_summary.loc[
    (['accepted', 'partial_pooling_20'], key_regions),
    [
        'rows',
        'observed_repair_share',
        'mean_repair_probability',
        'accuracy',
        'repair_recall',
        'expert_coverage',
    ],
]